In [4]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder

train = pd.read_csv("../Data/Processed/train.csv")
test = pd.read_csv("../Data/Processed/test.csv")
print(train.shape, test.shape)

(51200, 13) (12800, 13)


In [2]:
TREATMENT_COL = "segment"
OUTCOME_COL = "conversion"
CONTROL = "No E-Mail"
ID_COL = "Customer ID"

FEATURE_COLS = ["recency", "history", "mens", "womens", "newbie", "history_floor", "zip_code", "channel"]

LGBM_PARAMS = dict(n_estimators=200, learning_rate=0.05, num_leaves=15, min_child_samples=100,
                   subsample=0.8, subsample_freq=1, colsample_bytree=0.8, random_state=42, verbose=-1)

cat_ohe = ["zip_code", "channel"]

In [ ]:
preprocessor=ColumnTransformer([
    ('history_log',FunctionTransformer(np.log1p,feature_names_out='one_to_one'),['history']),
    ('cat_ohe',OneHotEncoder(drop=["Rural", "Multichannel"],handle_unknown='ignore',sparse_output=False),cat_ohe)
],remainder='drop',verbose_feature_names_out=False
).set_output(transform='pandas')

In [7]:
class DMLPipeline:
    
    def __init__(self, preprocessor, estimator, feature_cols, treatment_cols, outcome_col, control):
        self.preprocessor = preprocessor
        self.estimator = estimator
        self.feature_cols = feature_cols
        self.treatment_cols = treatment_cols
        self.outcome_col = outcome_col
        self.control = control
        
    def fit(self, df):
        X = self.preprocessor.fit_transform(df[self.feature_cols])    
        self.estimator.fit(Y=df[self.outcome_col].values, T=df[self.treatment_col].values, X=X.values)
        self.arms_ = [a for a in sorted(df[self.treatment_col].unique()) if a != self.control]
        self.feature_names_ = list(X.columns)
        return self